In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import re

In [2]:
df=pd.read_csv("Data\8_rent_data.csv")

C:\Users\USER\AppData\Local\Temp\ipykernel_29020\3778509836.py:1: DtypeWarning: Columns (56) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("Data\8_rent_data.csv")


# Khusi Observations

In [3]:
df1=df.copy()

In [4]:
# df1.head()

In [5]:
df1.columns

Index(['Unnamed: 0', 'Title', 'Rental_Price', 'Location', 'Bed', 'Baths',
       'Carpet Area', 'Floor', 'Status', 'Furnished Status', 'Rental Value',
       'Security Deposit', 'Address', 'Furnishing', 'link', 'Facing',
       'Age Of Construction', 'Flooring', 'Overlooking', 'Age of Construction',
       'Bath', 'Super Built-up Area', 'Developer', 'Project', 'Amenities',
       'Price', 'Price per sqft', 'Configuration', 'Tower & Unit', 'Beds',
       'Balcony', 'Balconies', 'Landmarks', 'Additional Rooms', 'Lifts',
       'Water Availability', 'Status of Electricity',
       'Floors allowed for construction', 'Lift', 'Car parking',
       'Covered Parking', 'Tower', 'Authority Approval', 'Unit',
       'Type of Ownership', 'RERA ID', 'Floors allowed', 'Plot Area',
       'Any Construction done', 'Boundary Wall', 'Transaction Type',
       'Additional Features', 'Approved', 'Floor allowed', 'No of Open Sides',
       'Dimensions(L X B)', 'Brokerage Response',
       'Floors Allowed F

In [6]:
# df1.transpose()

# **Basic Information about Data**
1. This Data has 58 columns and 24135 rows
2. This Data has no duplicated values

# **Column wise info**

1. In **`Rental_Price`** column around `2.04%` data has `'Call for Price'` and `'Call for Rent'` as a value otherwise both column has same information so we can drop `Rental_Price` and  in `Rental Value` column around `37.38%` rows has maintenance charge.

2. **`Location`** and **`Address`** column has same information about address but 'Address' column has more detailed information but i can't find any pattern in it so that we can get any info so i think we can drop 'Address' column as also 'Location' give enough info.As this column has 5372 unique values so during transformation one hot encoding may slow the model so from this may be we can group location wise and make a new column name like 'Average price' and also we can make a new coulumn as name 'City'(to validate the effect of City in price).

3. From **`Floor`** column we can extract two information like `current floor` and `total floor`

4. in **`Status`** we have two distinct information but majority(96%) is 'immediately' only 0.07% data is 'Legal & Infra Status' which give a different information so we can group this rows and nan value rows. 

5. here **`Facing`** column we can use ohe after converting the column.

6. Here second **`Age of Construction`** columns has more values and no contrary rows so we can drop first **`Age Of Construction`** and it has 7 unique values ['Under Construction','New Construction','Less than 5 years', '5 to 10 years', '10 to 15 years','15 to 20 years', 'Above 20 years'] so we can do bining here

7. in **`Overlooking`** column we can split this into this [Garden/Park,Main Road,Pool] information and then use one hot encoding.

8. In **`Configuration`** column we have information that the type of bhk available here and also what else thing present under this property.

9. In **`Type of Ownership`** column we have 4 unique values.

10. yes we can drop `Furnished Status` and `Furnishing` column has 3 unique values ['Unfurnished','Semi-Furnished', 'Furnished'] and we can do ordinal encoding here to transfer as they effect the price

11. Here in **`Authority Approval`** and **`Approval`** column has either both has same value or in one of this has null value so we can merge them  

12. Here **`Tower`** column has `99.19%` data missing so we can drop it.

13. **`RERA ID`** is basically a registration number for that project given by the state authority.As we have already the age of construction and this column has around `98%` missing values so we can remove it.

14. From **`Flooring`** column we get info about that by what kind of material floor is made up of.It has `304` unique values (no ordering) so we can use one hot encoding for transformation.

15. **`Price per sqft`** colums has a range of price value i think it give the appraximate price per carpet area 

16. In this data we have 3 types of area column `Super Built-up Area`(carpet area + all other usable area in flat),`Plot Area`(Total project area),`Carpet Area`(area which is usable) and we need to make each row of them in same unit

17. Here we have price type column `Rental Value`,`Sequrity Deposit`,`Price`,`Price per sqft`,`` 

18. From **`Covered parking`** and **`Car parking`** we can get two separate column name `Covered car parking`,`Open car parking`

19. Here `Dimension(L X B)` column has value in 6 rows and also it has no unit and in this 6 rows 'super built-up area' and 'Carpet Area' has nan value and only 'plot area' has values but no connection can be found between them. 

20. **`Water Availability`** column give info about how much time water is available and it has 5 unique timings like '1 Hour Available','2 Hour Available','6 Hour Available','12 Hour Available','24 Hour Available' so we can make this column numeric(like only 1,2,6,12,24).

# `Columns which need DataType Conversion`
1. `Rental_price`(object->float64)(after making all unit same)
2. `Bed`,`Baths`(float64->int32)
3. `Carpet Area`(object->float64)(after cleaning and make all unit same)
4. from `Floor` also we can get two int32 column

 

In [7]:
df1['Landmarks'].unique()

array([nan, 'nandgopal industrial estate',
       'the society is in d lane next to Chirag nagar police station. straight inside the lane at the end is Sai baba temple   the building is just there.  2 nd floor. flat no E 12',
       ..., 'Near D mart Nanded City, Pune',
       'Whispering Wind Phase 1, BanerPashan Link Road, Pune', 'Epitome'],
      dtype=object)

In [8]:
# df1[df1['Water Availability'].notna()]['Water Availability']

In [9]:
# df1['Transaction Type'].notna().value_counts()
df1['Transaction Type'].describe()
# df1[df1['Additional Features'].notna()]['Additional Features']

count         35
unique         2
top       Resale
freq          20
Name: Transaction Type, dtype: object

In [10]:
print(df1['Status'].unique())
print((df1['Status']=='Legal & Infra Status').mean()*100)
print((df1['Status']=='Immediately').mean()*100)

['Immediately' nan 'Legal & Infra Status']
0.07458048477315103
96.40356328982806


In [11]:
df1.describe()

,Unnamed: 0,Bed,Baths,Bath,Beds,Balcony,Balconies,Lifts,Floors allowed for construction,Lift,Covered Parking,Floors allowed,Floor allowed,No of Open Sides,Floors Allowed For Construction
count,24135.00000,3734.0,19552.000000,4294.0,20001.000000,8928.0,8286.000000,6341.000000,7152.000000,8218.000000,8313.000000,14.000000,2.0,12.000000,1.0
mean,12067.00000,1.0,2.683869,1.0,2.736513,1.0,2.414193,2.738369,5.777405,2.341324,2.578131,4.000000,1.0,3.000000,1.0
std,6967.31871,0.0,0.881936,0.0,0.828053,0.0,0.764677,1.170278,6.304864,1.260689,25.393215,1.797434,0.0,0.953463,NaN
min,0.00000,1.0,2.000000,1.0,2.000000,1.0,2.000000,2.000000,1.000000,1.000000,1.000000,2.000000,1.0,2.000000,1.0
25%,6033.50000,1.0,2.000000,1.0,2.000000,1.0,2.000000,2.000000,1.000000,2.000000,1.000000,3.000000,1.0,2.000000,1.0
50%,12067.00000,1.0,2.000000,1.0,3.000000,1.0,2.000000,2.000000,3.000000,2.000000,1.000000,3.000000,1.0,3.000000,1.0
75%,18100.50000,1.0,3.000000,1.0,3.000000,1.0,3.000000,3.000000,10.000000,3.000000,2.000000,4.750000,1.0,4.000000,1.0
max,24134.00000,1.0,10.000000,1.0,10.000000,1.0,10.000000,10.000000,20.000000,10.000000,901.000000,9.000000,1.0,4.000000,1.0


In [12]:

pd.set_option('display.max_colwidth', None)
df1['Super Built-up Area'].describe()

count                                                                                                                                           8289
unique                                                                                                                                          4978
top       600sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham₹17/sqft
freq                                                                                                                                              24
Name: Super Built-up Area, dtype: object

In [13]:
# df1[['Covered Parking','Car parking']]

In [14]:
df1[['Price','Price','Rental_Price']].head()


,Price,Price,Rental_Price
0,NaN,NaN,"₹45,000"
1,NaN,NaN,"₹33,000"
2,₹ 76.0 Lac - ₹ 2.75 Cr,₹ 76.0 Lac - ₹ 2.75 Cr,"₹44,000"
3,NaN,NaN,"₹90,000"
4,₹ 3.0 Cr - ₹ 3.67 Cr,₹ 3.0 Cr - ₹ 3.67 Cr,₹1.1 Lac


In [15]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
 
# df1[['Dimensions(L X B)','Super Built-up Area']]
df1.loc[df1['Dimensions(L X B)'].notna(), ['Dimensions(L X B)', 'Super Built-up Area','Carpet Area','Plot Area']]

,Dimensions(L X B),Super Built-up Area,Carpet Area,Plot Area
13109,60 X 40,NaN,NaN,267sqyrdsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham
13139,119 X 1190,NaN,NaN,120sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham
13197,40 X 54,NaN,NaN,240sqyrdsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham
14800,500 X 150,NaN,NaN,5acresqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham
15682,58 X 65,NaN,NaN,3200sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham
16885,1300 X 1300,NaN,NaN,1300sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham


In [16]:
df1['Dimensions(L X B)'].notna().sum()

np.int64(6)

In [17]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24135 entries, 0 to 24134
Data columns (total 58 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       24135 non-null  int64  
 1   Title                            24135 non-null  object 
 2   Rental_Price                     24135 non-null  object 
 3   Location                         24135 non-null  object 
 4   Bed                              3734 non-null   float64
 5   Baths                            19552 non-null  float64
 6   Carpet Area                      15572 non-null  object 
 7   Floor                            20497 non-null  object 
 8   Status                           23285 non-null  object 
 9   Furnished Status                 23831 non-null  object 
 10  Rental Value                     23642 non-null  object 
 11  Security Deposit                 16735 non-null  object 
 12  Address           

In [18]:
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# df1[['Super Built-up Area','Plot Area','Carpet Area']]

In [19]:
df1[['Location','Address']].sample()

,Location,Address
3840,"Garia, Kolkata","Garia, Kolkata - South 24 Parganas District, West Bengal"


In [20]:
df1[df1['Location'].isna()][['Location','Address']]

,Location,Address


In [21]:
df1[df1['Unit'].notna()]['Unit'].sample(3)

8235      , 60 Units
17197    , 295 Units
10151    , 500 Units
Name: Unit, dtype: object

In [22]:

df1['Boundary Wall'].unique()

array([nan, 'No', 'Yes'], dtype=object)

In [23]:
# (df1['Tower']==df1['Tower & Unit'])

In [24]:
(df1['Rental Value'].str.contains('maintenance', case=False, na=False).sum()/df1.shape[0])*100

np.float64(37.38553967267454)

In [25]:
df1[df1['Tower'].notna()].sample()['Tower']

16649    1 Tower
Name: Tower, dtype: object

In [26]:
(df1['Tower'].isna().sum()/df1.shape[0])*100

np.float64(99.19204474829087)

In [27]:
# df1['Floor']

In [28]:

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
 
print(df1.loc[df1['Tower'].notna(), ['Tower', 'Tower & Unit']])



            Tower Tower & Unit
32      15 Towers          NaN
181    136 Towers          NaN
240     15 Towers          NaN
263       1 Tower          NaN
334       1 Tower          NaN
467       1 Tower          NaN
738       1 Tower          NaN
964       1 Tower          NaN
1074      1 Tower          NaN
1090      1 Tower          NaN
1092      1 Tower          NaN
1160      1 Tower          NaN
1209      1 Tower          NaN
1213      1 Tower          NaN
1257     2 Towers          NaN
1303      1 Tower          NaN
1308      1 Tower          NaN
1310      1 Tower          NaN
1317      1 Tower          NaN
1320      1 Tower          NaN
1386      1 Tower          NaN
1810     3 Towers          NaN
1985      1 Tower          NaN
2040      1 Tower          NaN
2054      1 Tower          NaN
2103     3 Towers          NaN
2104      1 Tower          NaN
2177      1 Tower          NaN
2278      1 Tower          NaN
2302      1 Tower          NaN
2464      1 Tower          NaN
2474    

In [29]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df1.loc[df1['Plot Area'].notna(), ['Plot Area', 'Carpet Area','Super Built-up Area']]
( df1[['Plot Area','Carpet Area','Super Built-up Area']]
    .isna()
    .any(axis=1)
    .sum()
    / df1.shape[0]
) * 100

np.float64(100.0)

In [30]:
# Filter rows where one of both columns have values
first_values = df1[df1['Location'].notna() & ~df1['Address'].notna()]
second_values= df1[~df1['Location'].notna() & df1['Address'].notna()]
print(first_values.shape[0]/df1.shape[0]*100)
print(first_values.shape)
print(second_values.shape)

0.0
(0, 58)
(0, 58)


In [31]:
diff_rows = df1[
    (df1['Carpet Area'] != df1['Plot Area']) &
    (~df1['Carpet Area'].isna()) &
    (~df1['Plot Area'].isna())
]
print(diff_rows.shape)


(0, 58)


In [32]:
df1['Furnishing'].unique()

array(['Semi-Furnished', 'Furnished', 'Unfurnished', nan], dtype=object)

In [33]:
((df1[(df1['Rental_Price']=='Call for Price') | (df1['Rental_Price']=='Call for Rent')].shape[0])/(df1.shape[0]))*100

2.042676610731303

In [34]:
df1.columns.duplicated().any()

np.False_

In [35]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24135 entries, 0 to 24134
Data columns (total 58 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       24135 non-null  int64  
 1   Title                            24135 non-null  object 
 2   Rental_Price                     24135 non-null  object 
 3   Location                         24135 non-null  object 
 4   Bed                              3734 non-null   float64
 5   Baths                            19552 non-null  float64
 6   Carpet Area                      15572 non-null  object 
 7   Floor                            20497 non-null  object 
 8   Status                           23285 non-null  object 
 9   Furnished Status                 23831 non-null  object 
 10  Rental Value                     23642 non-null  object 
 11  Security Deposit                 16735 non-null  object 
 12  Address           

In [36]:

missing_percent = df1.isna().mean()*100
print(missing_percent)
print(missing_percent['Super Built-up Area'])
print(missing_percent['Plot Area'])
print(missing_percent['Carpet Area'])
print(missing_percent['Additional Features'])

Unnamed: 0                          0.000000
Title                               0.000000
Rental_Price                        0.000000
Location                            0.000000
Bed                                84.528693
Baths                              18.989020
Carpet Area                        35.479594
Floor                              15.073545
Status                              3.521856
Furnished Status                    1.259582
Rental Value                        2.042677
Security Deposit                   30.660866
Address                             0.000000
Furnishing                          1.230578
link                                0.000000
Facing                             32.169049
Age Of Construction                44.793868
Flooring                           50.743733
Overlooking                        36.370416
Age of Construction                21.214005
Bath                               82.208411
Super Built-up Area                65.655687
Developer 

# Swarnava Observations

In [37]:
df2=df.copy()

In [38]:
df2.columns

Index(['Unnamed: 0', 'Title', 'Rental_Price', 'Location', 'Bed', 'Baths',
       'Carpet Area', 'Floor', 'Status', 'Furnished Status', 'Rental Value',
       'Security Deposit', 'Address', 'Furnishing', 'link', 'Facing',
       'Age Of Construction', 'Flooring', 'Overlooking', 'Age of Construction',
       'Bath', 'Super Built-up Area', 'Developer', 'Project', 'Amenities',
       'Price', 'Price per sqft', 'Configuration', 'Tower & Unit', 'Beds',
       'Balcony', 'Balconies', 'Landmarks', 'Additional Rooms', 'Lifts',
       'Water Availability', 'Status of Electricity',
       'Floors allowed for construction', 'Lift', 'Car parking',
       'Covered Parking', 'Tower', 'Authority Approval', 'Unit',
       'Type of Ownership', 'RERA ID', 'Floors allowed', 'Plot Area',
       'Any Construction done', 'Boundary Wall', 'Transaction Type',
       'Additional Features', 'Approved', 'Floor allowed', 'No of Open Sides',
       'Dimensions(L X B)', 'Brokerage Response',
       'Floors Allowed F

In [39]:
df2.head(3)

,Unnamed: 0,Title,Rental_Price,Location,Bed,Baths,Carpet Area,Floor,Status,Furnished Status,Rental Value,Security Deposit,Address,Furnishing,link,Facing,Age Of Construction,Flooring,Overlooking,Age of Construction,Bath,Super Built-up Area,Developer,Project,Amenities,Price,Price per sqft,Configuration,Tower & Unit,Beds,Balcony,Balconies,Landmarks,Additional Rooms,Lifts,Water Availability,Status of Electricity,Floors allowed for construction,Lift,Car parking,Covered Parking,Tower,Authority Approval,Unit,Type of Ownership,RERA ID,Floors allowed,Plot Area,Any Construction done,Boundary Wall,Transaction Type,Additional Features,Approved,Floor allowed,No of Open Sides,Dimensions(L X B),Brokerage Response,Floors Allowed For Construction
0,0,1 BHK 500 Sq-ft Flat/Apartment For Rent in,"₹45,000","Military Road, Mumbai",1.0,2.0,500sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham₹90/sqft,5(Out of 7 Floors),Immediately,Semi-Furnished,"₹45,000",₹1.5 Lac,"Military Road, Mumbai - Western Mumbai, Maharashtra",Semi-Furnished,https://www.magicbricks.com/propertyDetails/1-BHK-500-Sq-ft-Multistorey-Apartment-FOR-Rent-Military-Road-in-Mumbai&id=4d423833373930373735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1 BHK 450 Sq-ft Flat/Apartment For Rent in,"₹33,000","Kannamwar Nagar 1, Mumbai",1.0,2.0,370sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham₹89/sqft,5(Out of 22 Floors),NaN,Semi-Furnished,"₹33,000",NaN,"Peak 25, Kannamwar Nagar 1, Mumbai - Central Mumbai, Maharashtra",Semi-Furnished,https://www.magicbricks.com/propertyDetails/1-BHK-450-Sq-ft-Multistorey-Apartment-FOR-Rent-Kannamwar-Nagar-1-in-Mumbai&id=4d423833373635353033,West,Less than 5 years,Normal Tiles/Kotah Stone,"Garden/Park, Main Road",Less than 5 years,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,"1 BHK Flat 650 Sq-ft For Rent in Satellite Garden,","₹44,000","Goregaon East, Mumbai",1.0,NaN,NaN,6(Out of 7 Floors),Immediately,Furnished,"₹44,000",₹1.5 Lac,"Goregaon East, Mumbai - Western Mumbai, Maharashtra",Furnished,https://www.magicbricks.com/propertyDetails/1-BHK-650-Sq-ft-Multistorey-Apartment-FOR-Rent-Goregaon-East-in-Mumbai&id=4d423833373838393935,North - East,5 to 10 years,NaN,Garden/Park,5 to 10 years,1.0,650sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham₹68/sqft,Satellite Builders Pvt. Ltd.,Satellite Garden,"Fire Fighting Equipment,Cycling & Jogging Track,Kids Play Area,Indoor Squash & Badminton Courts,Indoor Games Room,Flower Gardens,RO Water System,DTH Television Facility,Internet/Wi-Fi Connectivity,Intercom Facility,Air Conditioned,Vaastu Compliant,Private Terrace/Garden,Water Storage,Security,Reserved Parking,Park,Club House,Rain Water Harvesting,Lift,Power Back Up",₹ 76.0 Lac - ₹ 2.75 Cr,"₹ 13,818 - ₹ 18,966","1, 2 BHK Flats","13 Towers , 120 Units",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Himanshu Observation

### Column info 
1. `Title` column contain rental property info with some highlighted details 
2. `Rental price` column has price of the property which is shown in the website. The unit type are different so need conversion .
3. `Location` column has locality/location details with city name (some has only locality info)
4. `Bed` and `Beds` columns contain Bed number . It should be equal to the "# BHK" in the `Title` column . `Bed` and `Beds` can be merged. Both has *float* data type.
5. `Carpet area` column has the carpet area details in the starting with the mixed unit . From it I have to construct new column `CARPET_AREA` which contains the values in correct unit . And in the last part of the string contains price/sqft of that property.
6. `Baths` and `Bath` column contain the number bathrooms in the rental house . it is in float type.`Bath and Baths` can be merged.
7. `Status` column has two different values . I think this column employs the availability of the property .
15. `Furnishing` column has 7 more non nan value than `Furnished status` . if both column has value then they are same . drop `Furnished status`.
8. `link` column is nothing special can be dropped .
9. `Facing` column has value *south-facing* it can be converted to *south* and same for *east facing*.
10. `Age Of Construction` column has three values `under construction,newly constructed,less than 5 years` . need appropriate binning.  **there has two age of construction column drop first one**. 2nd one has more non nan values . and 2nd one has all the values of the first one also,no data discrepancy .
11. `Overlooking` values can be split in the new column and filled by group by and it can work as a one hot encoding.
12. `Balcony` and `Balconies` column contain the number balconies in the rental house . it is in float type.`Balcony and Balconies` can be merged.
13. `Rental Value` contains the `rental price` and some has monthly/yearly/per sqft maintenance . need to make a new column name `Maintenance Charge`.
14. `Security Deposit` column's unit need to change to make it same scale
16. `Super built area` and `carpet` and `Plot Area` are in complement like if one column has value then other column does not have value .
17. There are two `Floors allowed for construction ` column last one's value will be on earlier column.
18. `Developer` column has the name of construction group/person name 
19. `Project` column has name of the project under which the property was built
20. `Amenities` column has name of special highlighted things that the property is offering . some feature engineering is needed so that some points can be deleted if for that special offer has different column such as lift . or make more column from this column
21. `Price` column has total price of that flat/house.
22. `Configuration` this column has mainly what type of BHK has under the project in which this property is one of them. 
23. I am not able to find any meaning of `Tower & Unit` column data.
24. `Additional Rooms` column has info of different rooms in that property . Can make a new column based on number of rooms/split room type and do missing indicator filling.
25. `Lifts` and `Lift` column contain the number Lifts in the rental house . it is in float type.`Lift and Lifts` can be merged.
26. `Water Availability` column has total hours when the water is available . need feature engineering
27. `Status of Electricity` column has total hour of power cut . can do binning.
28. `Floors allowed for construction` column has maximum number of floors can be built .means in future many floors maximum can be built.
29. `Car parking` and `Covered Parking` columns data are mostly aligned . like if `car parking` has data 3 covered then `covered parking` also has data 3 . but in some cases there has data in `covered parking` but no covered info in `car parking`.
30. `Tower` 
31. `Floors allowed` and `Floor allowed` are in complement if one column has value then other has nan .
32. **Floors allowed and Floors allowed for construction** column mainly contain same info (Hypothesis only ).
33. `Authority Approval` and `Approved` has relation, like `approved` column might be short form of some names of `Authority Approval` column. Need feature engineering. `Authority Approval` column values are name of authority which has given the approval of that property.
34. `Floor` column contains the info of that property that how many floors that flat has and in which floor is available for rent 
35.  `Towers & Units` and `Tower` and `Unit` can be merged or  `Tower & Unit` can be split into `Unit` and `Tower` column . they are disjoint .
36. `Type of Ownership` column has info what kind of ownership that property has
37. `RERA ID` has id of that place where the property is located . Not important 
38. `Any Construction done` column has yes no value . it means small construction is done on that property.
39. `Boundary Wall` column has yes no value means that property has wall or not.
40. `Transaction Type` has two different non null values(rent and resale) . and the data is very less (37 rows has data only)
41. `Additional Features` has very less data. this data has info about construction or has info like "gated colony".
42. `No of Open Sides` has very less data (12) . it has three different values 2 or 3 or 4 
43. Drop `Dimensions(L X B)` column it has info of the `Super built area` column only in length X Breadth
44. `Brokerage Response` has only one value .Drop it.

In [40]:
df3=df.copy()

In [41]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24135 entries, 0 to 24134
Data columns (total 58 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       24135 non-null  int64  
 1   Title                            24135 non-null  object 
 2   Rental_Price                     24135 non-null  object 
 3   Location                         24135 non-null  object 
 4   Bed                              3734 non-null   float64
 5   Baths                            19552 non-null  float64
 6   Carpet Area                      15572 non-null  object 
 7   Floor                            20497 non-null  object 
 8   Status                           23285 non-null  object 
 9   Furnished Status                 23831 non-null  object 
 10  Rental Value                     23642 non-null  object 
 11  Security Deposit                 16735 non-null  object 
 12  Address           

## Merging and Droping columns

In [42]:
# droping Unnamed: 0 
df3=df3.drop(['Unnamed: 0'],axis=1)

In [43]:
# new column Total_Beds
df3['Total_Beds']=np.where(df3['Beds'].notna(),df3['Beds'],df3['Bed'])
#droping Beds and Bed 
df3=df3.drop(['Beds','Bed'],axis=1)

In [44]:
#new column Carpet_Area
df3['Carpet_Area']=np.where(df3['Carpet Area'].notna(),df3['Carpet Area'],np.where(df3['Super Built-up Area'].notna(),df3['Super Built-up Area'],df3['Plot Area']))
# droping Carpet Area ,Super Built-up Area,'Plot Area'
df3=df3.drop(['Carpet Area','Super Built-up Area','Plot Area'],axis=1)

In [45]:
# new column Total_Baths
df3['Total_Baths']=np.where(df3['Baths'].notna(),df3['Baths'],df3['Bath']) 
#droping Baths Bath
df3=df3.drop(['Baths','Bath'],axis=1)  

In [46]:
# new column Total_Balcony
df3['Total_Balcony']=np.where(df3['Balconies'].notna(),df3['Balconies'],df3['Balcony'])
# droping Balconies and Balcony
df3=df3.drop(['Balconies','Balcony'],axis=1)

In [47]:
# droping Furnished Status
df3=df3.drop(['Furnished Status'],axis=1)

In [48]:
# droping Age Of Construction	
df3= df3.drop(['Age Of Construction'],axis=1)

In [49]:
# new column Floors_allowed_for_construction
df3['Floors_allowed_for_construction']=np.where(df3['Floors allowed for construction'].notna(),df3['Floors allowed for construction'],df3['Floors Allowed For Construction'])
# droping Floors allowed for construction and Floors Allowed For Construction
df3=df3.drop(['Floors allowed for construction','Floors Allowed For Construction'],axis=1)

In [50]:
# new column Total_Floors_allowed
df3['Total_Floors_allowed']=np.where(df3['Floors allowed'].notna(),df3['Floors allowed'],df3['Floor allowed'])
# droping Floors allowed and Floor allowed
df3=df3.drop(['Floors allowed','Floor allowed'],axis=1)

1. If `Total_Floors_allowed` column contains a data then it is same as `Floors_allowed_for_construction`.

In [51]:
# droping Total_Floors_allowed column
df3=df3.drop(['Total_Floors_allowed'],axis=1)

In [52]:
# concate Tower and Unit column
df3['New Tower & Unit']=df3['Tower']+df3['Unit']
# new column Total_Tower & Unit
df3['Total_Tower & Unit']=np.where(df3['Tower & Unit'].notna(),df3['Tower & Unit'],df3['New Tower & Unit'])
# droping Tower & Unit,New Tower & Unit,Tower and Unit
df3=df3.drop(['Tower & Unit','New Tower & Unit','Tower','Unit'],axis=1)


In [53]:
# new column Total_Lifts
df3['Total_Lifts']=np.where(df3['Lifts'].notna(),df3['Lifts'],df3['Lift'])
# droping Lifts and Lift
df3=df3.drop(['Lifts','Lift'],axis=1)

In [54]:
# droping Dimensions(L X B)
df3=df3.drop(['Dimensions(L X B)'],axis=1)

In [55]:
# droping link
df3=df3.drop(['link'],axis=1)

## Making new column and modifing data in the columns

In [56]:
df31=df3.copy()

In [57]:
# converting the inconsistant data of Rental_Price and Security Deposit column
def convert_rent(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower()
    
    # remove ₹, commas, and ALL spaces between digits
    x = x.replace('₹', '')
    x = re.sub(r'\s+', '', x)   # remove all spaces
    x = x.replace(',', '')
    
    try:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        
        if 'cr' in x:
            value *= 10000000
        elif 'lac' in x or 'lakh' in x:
            value *= 100000
        
        return int(value)
    
    except:
        return np.nan

# apply transformation
df31['Rental_Price'] = df31['Rental_Price'].apply(convert_rent).astype('Int64')
df31['Security Deposit']=df31['Security Deposit'].apply(convert_rent).astype('Int64')


In [58]:
# making new column city from the column Location
df31['city'] = df31['Location'].astype(str).apply(lambda x: x.split(',')[-1].strip() if ',' in x else x.strip())

# droping Location column
df31.drop(['Location'],axis=1,inplace=True)

In [59]:
# creating new columns rental_floor and total_floor from Floor
def split_floor(x):
    if pd.isna(x):
        return pd.Series([np.nan, np.nan])
    
    x = str(x).lower()
    
    if 'ground' in x:
        return pd.Series([0, np.nan])
    if 'basement' in x:
        return pd.Series([-1, np.nan])
    
    nums = re.findall(r'\d+', x)
    
    if len(nums) >= 2:
        return pd.Series([int(nums[0]), int(nums[1])])
    elif len(nums) == 1:
        return pd.Series([int(nums[0]), np.nan])
    else:
        return pd.Series([np.nan, np.nan])

# create columns
df31[['rental_floor', 'total_floor']] = df31['Floor'].apply(split_floor)

# convert to Int16 (nullable integer)
df31['rental_floor'] = df31['rental_floor'].astype('Int16')
df31['total_floor'] = df31['total_floor'].astype('Int16')

# droping Floor column
df31.drop(['Floor'],axis=1,inplace=True)

In [60]:
# making new column maintenance(₹) from Rental value column
def extract_maintenance(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower()
    
    parts = x.split('|')
    
    if len(parts) > 1:
        return parts[1].strip()   # keep full part
    
    return np.nan

df31['maintenance(₹)'] = df31['Rental Value'].apply(extract_maintenance)

# droping Rental Value column
df31.drop(['Rental Value'],axis=1,inplace=True)

In [61]:
# changing data formats in Facing column

# Convert to lowercase for consistency
df31['Facing'] = df31['Facing'].str.lower()

# Replace variations
df31['Facing'] = df31['Facing'].replace({'south facing': 'south','east facing': 'east',})

# Optional: remove extra spaces
df31['Facing'] = df31['Facing'].str.strip()

In [62]:
# making new Carpet_area(sqft) and Carpet_area_price/sqft column from Carpet_Area

# ---- Unit conversion to sqft ----
unit_to_sqft = {'sqft': 1,'sqm': 10.7639,'sqyrd': 9,'acre': 43560,
    'bigha': 17452,   # approx (varies regionally)
    'hectare': 107639,'marla': 272.25,'kanal': 5445,'biswa': 1350,
    'ground': 2400,'cent': 435.6,'perch': 272.25,'guntha': 1089,
    'are': 1076.39,'katha': 720,'gaj': 9,'killa': 43560
}

def extract_area(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower()
    
    # Extract first number + unit
    match = re.search(r'(\d+\.?\d*)\s*(sqft|sqm|sqyrd|acre|bigha|hectare|marla|kanal|biswa|ground|cent|perch|guntha|are|katha|gaj|killa)', x)
    
    if match:
        value = float(match.group(1))
        unit = match.group(2)
        return value * unit_to_sqft.get(unit, np.nan)
    
    return np.nan


# ---- Extract price per sqft ----
def extract_price(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower()

    # remove commas first
    x = x.replace(',', '')
    
    match = re.search(r'₹\s*(\d+\.?\d*)\s*/\s*(sqft|sqm|sqyrd|acre|bigha|hectare|marla|kanal|biswa|ground|cent|perch|guntha|are|katha|gaj|killa)', x)
    
    if match:
        price = float(match.group(1))
        unit = match.group(2)
        
        # Convert to per sqft
        return price / unit_to_sqft.get(unit, 1)
    
    return np.nan


# ---- Apply functions ----
df31['Carpet_area(sqft)'] = df31['Carpet_Area'].apply(extract_area)
df31['Carpet_area_price/sqft'] = df31['Carpet_Area'].apply(extract_price)

# droping Carpet_Area column
df31.drop(['Carpet_Area'],axis=1,inplace=True)


In [63]:
# making new column Total_Property_Price(ave) from Price column
def convert_price_range(x):
    if pd.isna(x):
        return np.nan
    
    # Remove ₹ and spaces
    x = x.replace('₹', '').strip()
    
    # Split range
    parts = x.split('-')
    
    values = []
    
    for part in parts:
        part = part.strip()
        
        # Extract number and unit
        match = re.search(r'([\d\.]+)\s*(Lac|Cr)', part, re.IGNORECASE)
        
        if match:
            num = float(match.group(1))
            unit = match.group(2).lower()
            
            if unit == 'lac':
                values.append(num * 1e5)   # 1 Lac = 100,000
            elif unit == 'cr':
                values.append(num * 1e7)   # 1 Cr = 10,000,000
    
    # If only one value
    if len(values) == 1:
        return values[0]
    
    # If range → take average
    elif len(values) == 2:
        return sum(values) / 2
    
    return np.nan

# Apply to your dataframe
df31['Total_Property_Price(ave)'] = df31['Price'].apply(convert_price_range)

# droping Price column
df31.drop(['Price'],axis=1,inplace=True)

In [64]:
# making new column Total_Property_Price/sqft(ave) from Price per sqft column 
def convert_price_per_sqft(x):
    if pd.isna(x):
        return np.nan
    
    # Remove ₹ and commas
    x = x.replace('₹', '').replace(',', '').strip()
    
    # Split range
    parts = x.split('-')
    
    values = []
    
    for part in parts:
        part = part.strip()
        
        # Extract numeric value
        match = re.search(r'(\d+\.?\d*)', part)
        
        if match:
            values.append(float(match.group(1)))
    
    # Single value
    if len(values) == 1:
        return values[0]
    
    # Range → average
    elif len(values) == 2:
        return sum(values) / 2
    
    return np.nan

# Apply to dataframe
df31['Total_Property_Price/sqft(ave)'] = df31['Price per sqft'].apply(convert_price_per_sqft)

# droping Price per sqft column
df31.drop(['Price per sqft'],axis=1,inplace=True)

In [65]:
# making abbreviation of Authority Approval column data 

# Step 1: Define mapping
mapping = {
    'Developer': 'DEV',
    'City Municipal Corporation': 'CMC',
    'RWA/Co-operative Housing Society': 'RWA/CHS',
    'Bangalore Development Authority': 'BDA',
    'Bangalore International Airport Area Planning Authority': 'BIAAPA',
    'Kolkata Municipal Corporation': 'KMC',
    'Development Authority': 'DA',
    'Greater Hyderabad Municipal Corporation': 'GHMC',
    'Hyderabad Metropolitan Development Authority': 'HMDA'
}

# Step 2: Function to convert values
def convert_authority(val):
    if pd.isna(val):
        return val
    
    parts = [p.strip() for p in val.split(',')]
    abbr = [mapping.get(p, p) for p in parts]  # fallback if not found
    
    return ', '.join(abbr)

# Step 3: Apply to column
df31['Authority Approval'] = df31['Authority Approval'].apply(convert_authority)

df31['Authority Approval'] =np.where(df31['Authority Approval'].notna(),df31['Authority Approval'],df31['Approved'])

#droping Approved column
df31.drop(['Approved'],axis=1,inplace=True)

In [66]:
# making new column open_parking from column Car parking
def extract_parking(val):
    if pd.isna(val):
        return pd.Series([np.nan, np.nan])
    
    val = str(val)
    
    # Find numbers before "Covered" and "Open"
    covered_match = re.search(r'(\d+)\s*Covered', val, re.IGNORECASE)
    open_match = re.search(r'(\d+)\s*Open', val, re.IGNORECASE)
    
    covered = int(covered_match.group(1)) if covered_match else 0
    open_p = int(open_match.group(1)) if open_match else 0
    
    return pd.Series([covered, open_p])

df31[['covered_parking', 'open_parking']] = df31['Car parking'].apply(extract_parking)

df31['Covered Parking']=np.where(df31['Covered Parking'].notna(),df31['Covered Parking'],df31['covered_parking'])

# droping Car parking and covered_parking
df31.drop(['Car parking','covered_parking'],axis=1,inplace=True)



In [67]:
# making new column total_towers and total_units from Total_Tower & Unit column
def extract_tower_unit(val):
    if pd.isna(val):
        return pd.Series([np.nan, np.nan])
    
    val = str(val)
    
    towers_match = re.search(r'(\d+)\s*Towers?', val, re.IGNORECASE)
    units_match = re.search(r'(\d+)\s*Units?', val, re.IGNORECASE)
    
    towers = int(towers_match.group(1)) if towers_match else np.nan
    units = int(units_match.group(1)) if units_match else np.nan
    
    return pd.Series([towers, units])

df31[['total_towers', 'total_units']] = df31['Total_Tower & Unit'].apply(extract_tower_unit)

# droping Total_Tower & Unit column
df31.drop(['Total_Tower & Unit'],axis=1,inplace=True)

In [68]:
df31.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24135 entries, 0 to 24134
Data columns (total 44 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Title                            24135 non-null  object 
 1   Rental_Price                     23642 non-null  Int64  
 2   Status                           23285 non-null  object 
 3   Security Deposit                 16735 non-null  Int64  
 4   Address                          24135 non-null  object 
 5   Furnishing                       23838 non-null  object 
 6   Facing                           16371 non-null  object 
 7   Flooring                         11888 non-null  object 
 8   Overlooking                      15357 non-null  object 
 9   Age of Construction              19015 non-null  object 
 10  Developer                        9994 non-null   object 
 11  Project                          11666 non-null  object 
 12  Amenities         

In [69]:
df31.sample(5).transpose()

,13547,4482,3697,13615,21405
Title,"2 BHK Flat 1040 Sq-ft For Rent in PVR Enclave,",2 BHK Flat 1000 Sq-ft For Rent in,2 BHK 350 Sq-ft Flat/Apartment For Rent in,1 BHK Residential House For Rent,2 BHK Flat 1300 Sq-ft For Rent in
Rental_Price,25000,14000,6000,7000,35000
Status,Immediately,Immediately,Immediately,Immediately,Immediately
Security Deposit,54000,14000,10000,<NA>,70000
Address,"Bachupally, Bachupally, Hyderabad - North, Andhra Pradesh","CA Block, Near Axis Mall, Action Area 1, Action Area 1, Kolkata - East, West Bengal","Dum Dum, Kolkata - North, West Bengal","Attapur, Hyderabad - Central, Andhra Pradesh","Baner Pashan Link Road, Pune - West, Maharashtra"
Furnishing,Furnished,Semi-Furnished,Unfurnished,Semi-Furnished,Semi-Furnished
Facing,west,north,south -west,NaN,east
Flooring,NaN,Marble,NaN,NaN,NaN
Overlooking,NaN,NaN,"Garden/Park, Main Road",NaN,"Garden/Park, Main Road"
Age of Construction,10 to 15 years,Less than 5 years,Less than 5 years,NaN,Less than 5 years
